Data Source: https://data.cms.gov/provider-summary-by-type-of-service/medicare-inpatient-hospitals/medicare-inpatient-hospitals-by-provider-and-service

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio


c:\Users\khpha\anaconda3\lib\site-packages\pandas\core\computation\expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
c:\Users\khpha\anaconda3\lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (
c:\Users\khpha\anaconda3\lib\site-packages\scipy\__init__.py:155: UserWarning: A NumPy version >=1.18.5 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [5]:
# Ingest raw data
file_path = "data/MUP_INP_RY26_P03_V10_DY24_PrvSvc.csv"
raw_df = pd.read_csv(file_path, low_memory=False)
raw_df.head(5)

,Rndrng_Prvdr_CCN,Rndrng_Prvdr_Org_Name,Rndrng_Prvdr_City,Rndrng_Prvdr_St,Rndrng_Prvdr_State_FIPS,Rndrng_Prvdr_Zip5,Rndrng_Prvdr_State_Abrvtn,Rndrng_Prvdr_RUCA,Rndrng_Prvdr_RUCA_Desc,DRG_Cd,DRG_Desc,Tot_Dschrgs,Avg_Submtd_Cvrd_Chrg,Avg_Tot_Pymt_Amt,Avg_Mdcr_Pymt_Amt
0,10001,Southeast Health Medical Center,Dothan,1108 Ross Clark Circle,1,36301,AL,2.0,Metropolitan area high commuting: primary flow...,3,ECMO OR TRACHEOSTOMY WITH MV >96 HOURS OR PRIN...,11,738478.636360,103236.272730,91218.181818
1,10001,Southeast Health Medical Center,Dothan,1108 Ross Clark Circle,1,36301,AL,2.0,Metropolitan area high commuting: primary flow...,23,CRANIOTOMY WITH MAJOR DEVICE IMPLANT OR ACUTE ...,23,173562.086960,40220.217391,37634.565217
2,10001,Southeast Health Medical Center,Dothan,1108 Ross Clark Circle,1,36301,AL,2.0,Metropolitan area high commuting: primary flow...,24,CRANIOTOMY WITH MAJOR DEVICE IMPLANT OR ACUTE ...,13,95613.307692,27305.461538,25644.307692
3,10001,Southeast Health Medical Center,Dothan,1108 Ross Clark Circle,1,36301,AL,2.0,Metropolitan area high commuting: primary flow...,25,CRANIOTOMY AND ENDOVASCULAR INTRACRANIAL PROCE...,22,182831.409090,31420.954545,23270.863636
4,10001,Southeast Health Medical Center,Dothan,1108 Ross Clark Circle,1,36301,AL,2.0,Metropolitan area high commuting: primary flow...,38,EXTRACRANIAL PROCEDURES WITH CC,27,111318.555560,12183.925926,10532.222222


In [9]:
# Raw data overview
rows, cols = raw_df.shape
print(f"Shape: {rows} rows, {cols} columns")
print(f"Columns: {list(raw_df.columns)}")
overview_df = pd.DataFrame({
    "Column": raw_df.columns,
    "Dtype": raw_df.dtypes.astype(str),
    "Non-Null": raw_df.notnull().sum().values,
    "Nulls": raw_df.isnull().sum().values,
    "Null_%": (raw_df.isnull().sum().values / rows * 100).round(2),
    "Unique": raw_df.nunique().values
})
# Show all of overview_df without truncation
overview_df.head(len(overview_df))

Shape: 145879 rows, 15 columns
Columns: ['Rndrng_Prvdr_CCN', 'Rndrng_Prvdr_Org_Name', 'Rndrng_Prvdr_City', 'Rndrng_Prvdr_St', 'Rndrng_Prvdr_State_FIPS', 'Rndrng_Prvdr_Zip5', 'Rndrng_Prvdr_State_Abrvtn', 'Rndrng_Prvdr_RUCA', 'Rndrng_Prvdr_RUCA_Desc', 'DRG_Cd', 'DRG_Desc', 'Tot_Dschrgs', 'Avg_Submtd_Cvrd_Chrg', 'Avg_Tot_Pymt_Amt', 'Avg_Mdcr_Pymt_Amt']


,Column,Dtype,Non-Null,Nulls,Null_%,Unique
Rndrng_Prvdr_CCN,Rndrng_Prvdr_CCN,int64,145879,0,0.0,2906
Rndrng_Prvdr_Org_Name,Rndrng_Prvdr_Org_Name,object,145879,0,0.0,2845
Rndrng_Prvdr_City,Rndrng_Prvdr_City,object,145879,0,0.0,1762
Rndrng_Prvdr_St,Rndrng_Prvdr_St,object,145879,0,0.0,2897
Rndrng_Prvdr_State_FIPS,Rndrng_Prvdr_State_FIPS,int64,145879,0,0.0,51
Rndrng_Prvdr_Zip5,Rndrng_Prvdr_Zip5,int64,145879,0,0.0,2687
Rndrng_Prvdr_State_Abrvtn,Rndrng_Prvdr_State_Abrvtn,object,145879,0,0.0,51
Rndrng_Prvdr_RUCA,Rndrng_Prvdr_RUCA,float64,145879,0,0.0,19
Rndrng_Prvdr_RUCA_Desc,Rndrng_Prvdr_RUCA_Desc,object,145879,0,0.0,15
DRG_Cd,DRG_Cd,int64,145879,0,0.0,540


In [ ]:
# RUCA code distribution
raw_df["Rndrng_Prvdr_RUCA"].value_counts().sort_index()

Rndrng_Prvdr_RUCA
1.0     127031
1.1       2407
2.0       2243
2.1         36
3.0         27
4.0      10265
4.1        504
5.0        647
6.0         10
7.0       1335
7.1         60
7.2         78
8.0         75
8.2          3
9.0         10
10.0       482
10.1        87
10.3         1
99.0       578
Name: count, dtype: int64

In [14]:
# RUCA code distribution 

plot = px.histogram(raw_df[raw_df['Rndrng_Prvdr_RUCA'] < 99], x='Rndrng_Prvdr_RUCA', title='Distribution of RUCA Codes (Excluding code 99)', labels={'RUCA': 'RUCA Code'}, nbins=20)
plot.update_layout(bargap=0.1)
plot.show()

In [ ]:
# Data cleaning and preprocessing for pipeline

# Dropped columns: 

def select_and_rename(df):
    """
    Subsets relevant columns and renames to more readable/descriptive names based on mapping.
    """
    # Unused columns commented out, can be added back if needed
    column_mapping = {
            'Rndrng_Prvdr_CCN': 'provider_ccn',
            'Rndrng_Prvdr_Org_Name': 'provider_name',
            'Rndrng_Prvdr_City': 'city',
            #'Rndrng_Prvdr_St': 'provider_street_address',
            #'Rndrng_Prvdr_State_FIPS': 'provider_state_FIPS',
            #'Rndrng_Prvdr_Zip5': 'provider_zipcode',
            'Rndrng_Prvdr_State_Abrvtn': 'state',
            'Rndrng_Prvdr_RUCA': 'ruca', # RUCA for the zipcode where provider is physically
            'Rndrng_Prvdr_RUCA_Desc': 'ruca_desc',
            'DRG_Cd': 'drg_code',
            'DRG_Desc': 'drg_desc',
            'Tot_Dschrgs': 'total_discharges',
            'Avg_Submtd_Cvrd_Chrg': 'avg_charge',
            'Avg_Tot_Pymt_Amt': 'avg_payment',
            'Avg_Mdcr_Pymt_Amt': 'avg_medicare_payment'
        }
    kept_cols = [col for col in column_mapping.keys() if col in df.columns]
    
    return df[kept_cols].rename(columns=column_mapping)


def fix_dtypes(df):
    """
    Fix data types for specific columns 
    (provider_ccn and drg_code handled in subsequent function to pad identifiers)
    """
    df = df.copy()
    # Convert numeric columns to appropriate types
    numeric_columns = ['total_discharges', 'average_covered_charges', 'average_total_payments', 'average_medicare_payments']
    for col in numeric_columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    
    return df

def pad_identifiers(df):
    """
    Pad identifiers with leading zeros to ensure consistent length: 
    provider_ccn (6 digits), drg_code (3 digits)
    """
    df = df.copy()
    df['provider_ccn'] = df['provider_ccn'].astype(str).str.zfill(6)
    df['drg_code'] = df['drg_code'].astype(str).str.zfill(3)
    return df

def add_urbanicity_column(df): 
    """
    Add a new column 'urbanicity' based on the provider_ruca code.

    per Wikipedia, 1-3 is Metropolitan, 4-6 is Micropolitan, 7-10 is Small town/Rural.
    """
    df = df.copy()
    df['urbanicity'] = df['provider_ruca'].apply(lambda x: 'Metro' if x <= 3 else 'Non-Metro')
    return df



In [ ]:
cleaned_df = (
    raw_df
    .pipe(select_and_rename)
    .pipe(fix_dtypes)
    .pipe(pad_identifiers)
    .pipe(add_urbanicity_column)
)

In [17]:
cleaned_df.head()

,provider_ccn,provider_name,provider_city,provider_street_address,provider_state_FIPS,provider_zipcode,provider_state_abbr,provider_ruca,provider_ruca_desc,drg_code,drg_description,total_discharges,average_covered_charges,average_total_payments,average_medicare_payments,urbanicity
0,010001,Southeast Health Medical Center,Dothan,1108 Ross Clark Circle,1,36301,AL,2.0,Metropolitan area high commuting: primary flow...,003,ECMO OR TRACHEOSTOMY WITH MV >96 HOURS OR PRIN...,11,738478.636360,103236.272730,91218.181818,Metro
1,010001,Southeast Health Medical Center,Dothan,1108 Ross Clark Circle,1,36301,AL,2.0,Metropolitan area high commuting: primary flow...,023,CRANIOTOMY WITH MAJOR DEVICE IMPLANT OR ACUTE ...,23,173562.086960,40220.217391,37634.565217,Metro
2,010001,Southeast Health Medical Center,Dothan,1108 Ross Clark Circle,1,36301,AL,2.0,Metropolitan area high commuting: primary flow...,024,CRANIOTOMY WITH MAJOR DEVICE IMPLANT OR ACUTE ...,13,95613.307692,27305.461538,25644.307692,Metro
3,010001,Southeast Health Medical Center,Dothan,1108 Ross Clark Circle,1,36301,AL,2.0,Metropolitan area high commuting: primary flow...,025,CRANIOTOMY AND ENDOVASCULAR INTRACRANIAL PROCE...,22,182831.409090,31420.954545,23270.863636,Metro
4,010001,Southeast Health Medical Center,Dothan,1108 Ross Clark Circle,1,36301,AL,2.0,Metropolitan area high commuting: primary flow...,038,EXTRACRANIAL PROCEDURES WITH CC,27,111318.555560,12183.925926,10532.222222,Metro


In [18]:
cleaned_df['provider_state_abbr'].nunique() / cleaned_df.count() 

provider_ccn                 0.00035
provider_name                0.00035
provider_city                0.00035
provider_street_address      0.00035
provider_state_FIPS          0.00035
provider_zipcode             0.00035
provider_state_abbr          0.00035
provider_ruca                0.00035
provider_ruca_desc           0.00035
drg_code                     0.00035
drg_description              0.00035
total_discharges             0.00035
average_covered_charges      0.00035
average_total_payments       0.00035
average_medicare_payments    0.00035
urbanicity                   0.00035
dtype: float64

In [21]:

def get_recommended_categorical_columns(
    df: pd.DataFrame, threshold: float = 0.5
) -> list:
    """Evaluates all object/string columns in a DataFrame and returns a list

    of columns recommended for conversion to categorical.

    Args:
        df: The pandas DataFrame to evaluate.
        threshold: The maximum uniqueness ratio (unique/total) allowed
                   to recommend conversion. Default is 0.5 (50%).
    """
    recommended_cols = []

    # Select only object and string columns
    target_cols = df.select_dtypes(include=["object", "string", "category"]).columns

    for col in target_cols:
        # Skip if already categorical
        if isinstance(df[col].dtype, pd.CategoricalDtype):
            continue

        total_count = len(df[col])
        if total_count == 0:
            continue

        # Calculate cardinality ratio
        unique_count = df[col].nunique()
        cardinality_ratio = unique_count / total_count

        # Test deep memory usage for both states
        mem_object = df[col].memory_usage(deep=True)
        mem_cat = df[col].astype("category").memory_usage(deep=True)

        # Recommend if it passes the cardinality threshold AND saves memory
        if cardinality_ratio <= threshold and mem_cat < mem_object:
            recommended_cols.append(col)

    return recommended_cols

get_recommended_categorical_columns(cleaned_df)

['provider_ccn',
 'provider_name',
 'provider_city',
 'provider_street_address',
 'provider_state_abbr',
 'provider_ruca_desc',
 'drg_code',
 'drg_description',
 'urbanicity']